# Prompt Engineering Project
## Part 1: Setup & Initial Prompts

This notebook implements systematic prompt testing to diagnose failures and improve consistency.
- **Objective**: Test three task types (Sentiment Analysis, Product Description, Data Extraction)
- **Model**: GPT-4o mini (cost-effective)
- **Budget**: $2.00 USD total
- **Output**: TXT files with results and analysis

## 1A: Initialize Environment & OpenAI Client
Load API key from Windows environment variable, validate connection, and initialize client.

In [1]:
import os
import sys
from datetime import datetime
from pathlib import Path
import json
import tiktoken
from openai import OpenAI

# Validate OPENAI_API_KEY from Windows environment
api_key = os.getenv("OPENAI_API_KEY")
if not api_key or api_key.strip() == "":
    raise ValueError(
        "ERROR: OPENAI_API_KEY environment variable not found or empty.\n"
        "Please set it in Windows Environment Variables and restart the kernel."
    )

# Initialize OpenAI client (key is not stored, only client object)
client = OpenAI(api_key=api_key)

# Validate connection by listing models
try:
    models = client.models.list()
    print("✓ OpenAI API Connection Validated")
    print(f"✓ API Key loaded from environment variable")
except Exception as e:
    print(f"✗ Connection failed: {e}")
    raise

# Setup
MODEL = "gpt-4o-mini"  # Cost-effective, capable model
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

print(f"✓ Model: {MODEL}")
print(f"✓ Results directory: {RESULTS_DIR.absolute()}")

✓ OpenAI API Connection Validated
✓ API Key loaded from environment variable
✓ Model: gpt-4o-mini
✓ Results directory: c:\Users\kupit\week 2\d2\results


## 1B: Cost Tracking Setup
Initialize token counter and cost calculator for budget enforcement ($1 USD limit).

In [2]:
# Token encoding for cost calculation
encoding = tiktoken.encoding_for_model(MODEL)

# Cost per token (GPT-4o mini pricing as of 2024)
COST_PER_INPUT_TOKEN = 0.00003  # $0.03 per 1M input tokens
COST_PER_OUTPUT_TOKEN = 0.00006  # $0.06 per 1M output tokens
BUDGET_LIMIT_USD = 4.00  # $4 total budget ($2 base + $2 for PART 6)

# Global cost tracker
total_cost_usd = 0.0
total_input_tokens = 0
total_output_tokens = 0
api_calls_made = 0

def count_tokens(text):
    """Count tokens in text using tiktoken."""
    return len(encoding.encode(text))

def estimate_cost(input_tokens, output_tokens):
    """Estimate cost for a single API call."""
    cost = (input_tokens * COST_PER_INPUT_TOKEN) + (output_tokens * COST_PER_OUTPUT_TOKEN)
    return cost

def check_budget(estimated_cost):
    """Check if estimated cost would exceed budget."""
    global total_cost_usd
    if total_cost_usd + estimated_cost > BUDGET_LIMIT_USD:
        print(f"⚠ WARNING: Budget limit exceeded. Current: ${total_cost_usd:.4f}, "
              f"Estimated additional: ${estimated_cost:.4f}")
        return False
    return True

def log_cost(input_tokens, output_tokens):
    """Log cost from an API call."""
    global total_cost_usd, total_input_tokens, total_output_tokens, api_calls_made
    cost = estimate_cost(input_tokens, output_tokens)
    total_cost_usd += cost
    total_input_tokens += input_tokens
    total_output_tokens += output_tokens
    api_calls_made += 1
    return cost

def print_cost_summary():
    """Print current cost tracking summary."""
    print(f"\n--- COST TRACKING SUMMARY ---")
    print(f"Total API calls made: {api_calls_made}")
    print(f"Total input tokens: {total_input_tokens}")
    print(f"Total output tokens: {total_output_tokens}")
    print(f"Total cost: ${total_cost_usd:.6f} USD")
    print(f"Remaining budget: ${BUDGET_LIMIT_USD - total_cost_usd:.6f} USD")
    print(f"Budget status: {'✓ OK' if total_cost_usd <= BUDGET_LIMIT_USD else '✗ EXCEEDED'}")
    print()

print("✓ Cost tracking initialized")
print(f"✓ Budget limit: ${BUDGET_LIMIT_USD}")
print(f"✓ Input token cost: ${COST_PER_INPUT_TOKEN:.6f}")
print(f"✓ Output token cost: ${COST_PER_OUTPUT_TOKEN:.6f}")

✓ Cost tracking initialized
✓ Budget limit: $4.0
✓ Input token cost: $0.000030
✓ Output token cost: $0.000060


## 1C: Helper Functions for Prompt Testing
Create reusable functions for calling OpenAI API and running prompts multiple times.

In [3]:
def call_openai(prompt_text, temperature=0.7, max_tokens=500):
    """
    Call OpenAI API with a single prompt.
    
    Args:
        prompt_text: The prompt to send
        temperature: Sampling temperature (0.0 to 2.0)
        max_tokens: Maximum tokens in response
    
    Returns:
        response_text: The model's response
        cost: Cost of this call in USD
    """
    # Count input tokens
    input_tokens = count_tokens(prompt_text)
    
    # Estimate cost before calling API
    estimated_output_tokens = 200  # Conservative estimate
    estimated_cost = estimate_cost(input_tokens, estimated_output_tokens)
    
    if not check_budget(estimated_cost):
        raise RuntimeError(f"Budget would be exceeded. Estimated cost: ${estimated_cost:.6f}")
    
    try:
        # Make API call
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt_text}],
            temperature=temperature,
            max_tokens=max_tokens
        )
        
        response_text = response.choices[0].message.content
        actual_input_tokens = response.usage.prompt_tokens
        actual_output_tokens = response.usage.completion_tokens
        actual_cost = log_cost(actual_input_tokens, actual_output_tokens)
        
        return response_text, actual_cost
    
    except Exception as e:
        print(f"✗ API call failed: {e}")
        raise

def run_n_times(prompt_text, n_runs, output_filename, temperature=0.7, max_tokens=500):
    """
    Run a prompt n times and save all responses to a TXT file.
    
    Args:
        prompt_text: The prompt to test
        n_runs: Number of times to run
        output_filename: Name of output file in results/ directory
        temperature: Sampling temperature
        max_tokens: Maximum tokens per response
    
    Returns:
        responses_list: List of all responses
        total_run_cost: Total cost for all runs
    """
    output_path = RESULTS_DIR / output_filename
    responses_list = []
    total_run_cost = 0.0
    
    print(f"\n▶ Running prompt {n_runs} times: {output_filename}")
    
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(f"=== PROMPT TESTING RESULTS ===\n")
        f.write(f"Timestamp: {datetime.now().isoformat()}\n")
        f.write(f"Model: {MODEL}\n")
        f.write(f"Number of runs: {n_runs}\n")
        f.write(f"Temperature: {temperature}\n")
        f.write(f"\nPrompt:\n{'-' * 60}\n{prompt_text}\n{'-' * 60}\n\n")
        f.write(f"Responses:\n{'=' * 60}\n")
        
        for i in range(1, n_runs + 1):
            try:
                response, cost = call_openai(prompt_text, temperature=temperature, max_tokens=max_tokens)
                responses_list.append(response)
                total_run_cost += cost
                
                f.write(f"\n--- Response {i} (Cost: ${cost:.6f}) ---\n")
                f.write(f"{response}\n")
                
                print(f"  [{i}/{n_runs}] ✓ Cost: ${cost:.6f}, Remaining: ${BUDGET_LIMIT_USD - total_cost_usd:.6f}")
            
            except Exception as e:
                print(f"  [{i}/{n_runs}] ✗ Failed: {e}")
                f.write(f"\n--- Response {i} (ERROR) ---\n")
                f.write(f"Failed: {e}\n")
                break
        
        f.write(f"\n{'=' * 60}\n")
        f.write(f"Total cost for {len(responses_list)} responses: ${total_run_cost:.6f}\n")
    
    print(f"  Saved to: {output_path}")
    return responses_list, total_run_cost

print("✓ Helper functions created: call_openai(), run_n_times()")

✓ Helper functions created: call_openai(), run_n_times()


## 1D: Create Results Directory & Verify Setup
Ensure results directory exists and all systems are ready.

In [ ]:
# Verify results directory
if RESULTS_DIR.exists() and RESULTS_DIR.is_dir():
    print(f"✓ Results directory exists: {RESULTS_DIR.absolute()}")
else:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    print(f"✓ Results directory created: {RESULTS_DIR.absolute()}")

# Ready message
print("\n" + "=" * 60)
print("SETUP COMPLETE - READY FOR TESTING")
print("=" * 60)
print(f"✓ OpenAI Client: Initialized and validated")
print(f"✓ Model: {MODEL}")
print(f"✓ Budget: ${BUDGET_LIMIT_USD} USD")
print(f"✓ Results Directory: {RESULTS_DIR}")
print(f"✓ API Key: Loaded from OPENAI_API_KEY (not logged)")
print("\nReady to start prompt testing.")

✓ Results directory exists: c:\Users\kupit\week 2\d2\results

SETUP COMPLETE - READY FOR TESTING
✓ OpenAI Client: Initialized and validated
✓ Model: gpt-4o-mini
✓ Bug­get: $1.0 USD
✓ Results Directory: results
✓ API Key: Loaded from OPENAI_API_KEY (not logged)

Ready to start prompt testing.


## 1E: Test Sentiment Analysis Prompt v1
Run the initial sentiment analysis prompt once to verify it works.

In [5]:
# Initial simple prompt for sentiment analysis (from task description)
sentiment_prompt_v1 = """Classify this customer message: "I love this product! It's exactly what I needed."""

# Test it once
print("\n" + "=" * 60)
print("TASK 1: SENTIMENT ANALYSIS (v1)")
print("=" * 60)
print(f"Prompt: {sentiment_prompt_v1}")
print()

try:
    result, cost = call_openai(sentiment_prompt_v1)
    print("✓ Sentiment Analysis Result:")
    print(result)
    print(f"\nCost for this call: ${cost:.6f}")
except Exception as e:
    print(f"✗ Failed: {e}")


TASK 1: SENTIMENT ANALYSIS (v1)
Prompt: Classify this customer message: "I love this product! It's exactly what I needed.

✓ Sentiment Analysis Result:
The customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.

Cost for this call: $0.001770


## 1F: Test Product Description Prompt v1
Run the initial product description prompt once to verify it works.

In [6]:
# Initial simple prompt for product description (from task description)
product_prompt_v1 = """Create a product description for a wireless mouse that costs $29.99."""

# Test it once
print("\n" + "=" * 60)
print("TASK 2: PRODUCT DESCRIPTION (v1)")
print("=" * 60)
print(f"Prompt: {product_prompt_v1}")
print()

try:
    result, cost = call_openai(product_prompt_v1)
    print("✓ Product Description Result:")
    print(result)
    print(f"\nCost for this call: ${cost:.6f}")
except Exception as e:
    print(f"✗ Failed: {e}")


TASK 2: PRODUCT DESCRIPTION (v1)
Prompt: Create a product description for a wireless mouse that costs $29.99.

✓ Product Description Result:
**Product Name: SwiftClick Wireless Mouse**

**Price: $29.99**

**Description:**

Elevate your computing experience with the SwiftClick Wireless Mouse, where sleek design meets exceptional functionality. Priced at just $29.99, this mouse is the perfect companion for bot[... ]

**Key Features:**

- **Wireless Freedom**: Enjoy the convenience of a clutter-free workspace with our advanced 2.4GHz wireless technology. The SwiftClick connects effortlessly to your device, providing a rel[... ]

- **Ergonomic Design**: Crafted with user comfort in mind, the SwiftClick features a contoured shape that fits naturally in your hand, reducing strain during long hours of use. Its texture[... ]

- **Adjustable DPI Settings**: Tailor your cursor sensitivity with adjustable DPI settings (800/1200/1600), allowing for smooth transitions between tasks, whether you’re

## 1G: Test Data Extraction Prompt v1
Run the initial data extraction prompt once to verify it works.

In [7]:
# Initial simple prompt for data extraction (from task description)
extraction_prompt_v1 = """Extract information from this customer feedback: "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."""

# Test it once
print("\n" + "=" * 60)
print("TASK 3: DATA EXTRACTION (v1)")
print("=" * 60)
print(f"Prompt: {extraction_prompt_v1}")
print()

try:
    result, cost = call_openai(extraction_prompt_v1)
    print("✓ Data Extraction Result:")
    print(result)
    print(f"\nCost for this call: ${cost:.6f}")
except Exception as e:
    print(f"✗ Failed: {e}")


TASK 3: DATA EXTRACTION (v1)
Prompt: Extract information from this customer feedback: "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged.

✓ Data Extraction Result:
Here is the extracted information from the customer feedback:

- **Order Number**: #12345
- **Order Date**: March 15th
- **Delivery Speed**: Fast
- **Issue**: Damaged packaging

Cost for this call: $0.003930


## PART 1 CHECKPOINT
Summary of Part 1 completion - All three initial prompts have been tested once successfully.

In [8]:
print("\n" + "=" * 60)
print("PART 1 CHECKPOINT - SETUP COMPLETE")
print("=" * 60)
print()
print_cost_summary()
print("✓ All three prompt types have been tested")
print("✓ Cost tracking is operational")
print("✓ Results directory is ready")
print("\nReady for PART 2: Systematic Testing (5x, 10x, 15x iterations)")


PART 1 CHECKPOINT - SETUP COMPLETE


--- COST TRACKING SUMMARY ---
Total API calls made: 3
Total input tokens: 84
Total output tokens: 521
Total cost: $0.033780 USD
Remaining budget: $0.966220 USD
Budget status: ✓ OK

✓ All three prompt types have been tested
✓ Cost tracking is operational
✓ Results directory is ready

Ready for PART 2: Systematic Testing (5x, 10x, 15x iterations)


# PART 2: Systematic Testing - Failure Diagnosis
## Objective: Run each prompt multiple times (5, 10, 15 iterations) to identify failure patterns

In [9]:
## 2A: Run Sentiment Analysis 5 Times
print("\n" + "=" * 60)
print("2A: SENTIMENT ANALYSIS - 5 ITERATIONS")
print("=" * 60)

sentiment_5_results, sentiment_5_cost = run_n_times(
    sentiment_prompt_v1, 
    5, 
    "sentiment_5_runs.txt"
)

print(f"\n✓ Completed 5 runs for Sentiment Analysis")
print(f"✓ Total cost: ${sentiment_5_cost:.6f}")
print_cost_summary()


2A: SENTIMENT ANALYSIS - 5 ITERATIONS

▶ Running prompt 5 times: sentiment_5_runs.txt
  [1/5] ✓ Cost: $0.001770, Remaining: $0.964450
  [2/5] ✓ Cost: $0.001770, Remaining: $0.962680
  [3/5] ✓ Cost: $0.001710, Remaining: $0.960970
  [4/5] ✓ Cost: $0.001770, Remaining: $0.959200
  [5/5] ✓ Cost: $0.001770, Remaining: $0.957430
  Saved to: results\sentiment_5_runs.txt

✓ Completed 5 runs for Sentiment Analysis
✓ Total cost: $0.008790

--- COST TRACKING SUMMARY ---
Total API calls made: 8
Total input tokens: 209
Total output tokens: 605
Total cost: $0.042570 USD
Remaining budget: $0.957430 USD
Budget status: ✓ OK



In [10]:
## 2B: Run Product Description 5 Times
print("\n" + "=" * 60)
print("2B: PRODUCT DESCRIPTION - 5 ITERATIONS")
print("=" * 60)

product_5_results, product_5_cost = run_n_times(
    product_prompt_v1, 
    5, 
    "product_5_runs.txt"
)

print(f"\n✓ Completed 5 runs for Product Description")
print(f"✓ Total cost: ${product_5_cost:.6f}")
print_cost_summary()


2B: PRODUCT DESCRIPTION - 5 ITERATIONS

▶ Running prompt 5 times: product_5_runs.txt
  [1/5] ✓ Cost: $0.021240, Remaining: $0.936190
  [2/5] ✓ Cost: $0.022320, Remaining: $0.913870
  [3/5] ✓ Cost: $0.024780, Remaining: $0.889090
  [4/5] ✓ Cost: $0.024360, Remaining: $0.864730
  [5/5] ✓ Cost: $0.020100, Remaining: $0.844630
  Saved to: results\product_5_runs.txt

✓ Completed 5 runs for Product Description
✓ Total cost: $0.112800

--- COST TRACKING SUMMARY ---
Total API calls made: 13
Total input tokens: 319
Total output tokens: 2430
Total cost: $0.155370 USD
Remaining budget: $0.844630 USD
Budget status: ✓ OK



In [11]:
## 2C: Run Data Extraction 5 Times
print("\n" + "=" * 60)
print("2C: DATA EXTRACTION - 5 ITERATIONS")
print("=" * 60)

extraction_5_results, extraction_5_cost = run_n_times(
    extraction_prompt_v1, 
    5, 
    "extraction_5_runs.txt"
)

print(f"\n✓ Completed 5 runs for Data Extraction")
print(f"✓ Total cost: ${extraction_5_cost:.6f}")
print_cost_summary()


2C: DATA EXTRACTION - 5 ITERATIONS

▶ Running prompt 5 times: extraction_5_runs.txt
  [1/5] ✓ Cost: $0.003930, Remaining: $0.840700
  [2/5] ✓ Cost: $0.003690, Remaining: $0.837010
  [3/5] ✓ Cost: $0.003690, Remaining: $0.833320
  [4/5] ✓ Cost: $0.003690, Remaining: $0.829630
  [5/5] ✓ Cost: $0.003930, Remaining: $0.825700
  Saved to: results\extraction_5_runs.txt

✓ Completed 5 runs for Data Extraction
✓ Total cost: $0.018930

--- COST TRACKING SUMMARY ---
Total API calls made: 18
Total input tokens: 504
Total output tokens: 2653
Total cost: $0.174300 USD
Remaining budget: $0.825700 USD
Budget status: ✓ OK



In [16]:
## 2D: Analyze 5-Run Results
print("\n" + "=" * 60)
print("2D: ANALYSIS OF 5-RUN RESULTS")
print("=" * 60)

def analyze_consistency(responses_list, task_name):
    """Analyze consistency of responses."""
    print(f"\n{task_name}:")
    print(f"  Total responses: {len(responses_list)}")
    
    # Check for exact matches
    if len(responses_list) > 0:
        unique_responses = len(set(responses_list))
        consistency_pct = (1 - (unique_responses - 1) / len(responses_list)) * 100 if len(responses_list) > 1 else 100
        print(f"  Unique responses: {unique_responses}/{len(responses_list)}")
        print(f"  Consistency (exact match): {consistency_pct:.1f}%")
        
        # Show response lengths
        lengths = [len(r) for r in responses_list]
        print(f"  Response length range: {min(lengths)} - {max(lengths)} characters")
        print(f"  Average length: {sum(lengths) / len(lengths):.0f} characters")
    
    return consistency_pct if len(responses_list) > 0 else 0

sentiment_5_consistency = analyze_consistency(sentiment_5_results, "Sentiment Analysis (5×)")
product_5_consistency = analyze_consistency(product_5_results, "Product Description (5×)")
extraction_5_consistency = analyze_consistency(extraction_5_results, "Data Extraction (5×)")

print("\n✓ 5-run analysis completed")


2D: ANALYSIS OF 5-RUN RESULTS


NameError: name 'sentiment_5_results' is not defined

In [13]:
## 2E: Run All Prompts 10 Times
print("\n" + "=" * 60)
print("2E: RUNNING ALL PROMPTS 10 TIMES")
print("=" * 60)

print("\n▶ Starting 10-iteration tests...")

sentiment_10_results, sentiment_10_cost = run_n_times(
    sentiment_prompt_v1, 
    10, 
    "sentiment_10_runs.txt"
)
print(f"✓ Sentiment 10× complete. Cost: ${sentiment_10_cost:.6f}")

product_10_results, product_10_cost = run_n_times(
    product_prompt_v1, 
    10, 
    "product_10_runs.txt"
)
print(f"✓ Product 10× complete. Cost: ${product_10_cost:.6f}")

extraction_10_results, extraction_10_cost = run_n_times(
    extraction_prompt_v1, 
    10, 
    "extraction_10_runs.txt"
)
print(f"✓ Extraction 10× complete. Cost: ${extraction_10_cost:.6f}")

print_cost_summary()


2E: RUNNING ALL PROMPTS 10 TIMES

▶ Starting 10-iteration tests...

▶ Running prompt 10 times: sentiment_10_runs.txt
  [1/10] ✓ Cost: $0.001770, Remaining: $0.823930
  [2/10] ✓ Cost: $0.001770, Remaining: $0.822160
  [3/10] ✓ Cost: $0.001770, Remaining: $0.820390
  [4/10] ✓ Cost: $0.001770, Remaining: $0.818620
  [5/10] ✓ Cost: $0.001710, Remaining: $0.816910
  [6/10] ✓ Cost: $0.002370, Remaining: $0.814540
  [7/10] ✓ Cost: $0.000870, Remaining: $0.813670
  [8/10] ✓ Cost: $0.001770, Remaining: $0.811900
  [9/10] ✓ Cost: $0.001590, Remaining: $0.810310
  [10/10] ✓ Cost: $0.001710, Remaining: $0.808600
  Saved to: results\sentiment_10_runs.txt
✓ Sentiment 10× complete. Cost: $0.017100

▶ Running prompt 10 times: product_10_runs.txt
  [1/10] ✓ Cost: $0.025020, Remaining: $0.783580
  [2/10] ✓ Cost: $0.025500, Remaining: $0.758080
  [3/10] ✓ Cost: $0.024540, Remaining: $0.733540
  [4/10] ✓ Cost: $0.023280, Remaining: $0.710260
  [5/10] ✓ Cost: $0.023700, Remaining: $0.686560
  [6/10] ✓ Cos

In [14]:
## 2F: Analyze 10-Run Results & Compare with 5-Run
print("\n" + "=" * 60)
print("2F: ANALYSIS OF 10-RUN RESULTS (vs 5-run)")
print("=" * 60)

sentiment_10_consistency = analyze_consistency(sentiment_10_results, "Sentiment Analysis (10×)")
product_10_consistency = analyze_consistency(product_10_results, "Product Description (10×)")
extraction_10_consistency = analyze_consistency(extraction_10_results, "Data Extraction (10×)")

print("\n" + "=" * 60)
print("CONSISTENCY COMPARISON: 5× vs 10×")
print("=" * 60)
print(f"Sentiment:     5×: {sentiment_5_consistency:.1f}% | 10×: {sentiment_10_consistency:.1f}%")
print(f"Product:       5×: {product_5_consistency:.1f}% | 10×: {product_10_consistency:.1f}%")
print(f"Extraction:    5×: {extraction_5_consistency:.1f}% | 10×: {extraction_10_consistency:.1f}%")


2F: ANALYSIS OF 10-RUN RESULTS (vs 5-run)

Sentiment Analysis (10×):
  Total responses: 10
  Unique responses: 9/10
  Consistency (exact match): 20.0%
  Response length range: 17 - 154 characters
  Average length: 88 characters

Product Description (10×):
  Total responses: 10
  Unique responses: 10/10
  Consistency (exact match): 10.0%
  Response length range: 1816 - 2341 characters
  Average length: 2019 characters

Data Extraction (10×):
  Total responses: 10
  Unique responses: 7/10
  Consistency (exact match): 40.0%
  Response length range: 145 - 195 characters
  Average length: 177 characters

CONSISTENCY COMPARISON: 5× vs 10×
Sentiment:     5×: 60.0% | 10×: 20.0%
Product:       5×: 20.0% | 10×: 10.0%
Extraction:    5×: 20.0% | 10×: 40.0%


In [15]:
## 2G: Run All Prompts 15 Times
print("\n" + "=" * 60)
print("2G: RUNNING ALL PROMPTS 15 TIMES")
print("=" * 60)

print("\n▶ Starting 15-iteration tests...")

sentiment_15_results, sentiment_15_cost = run_n_times(
    sentiment_prompt_v1, 
    15, 
    "sentiment_15_runs.txt"
)
print(f"✓ Sentiment 15× complete. Cost: ${sentiment_15_cost:.6f}")

product_15_results, product_15_cost = run_n_times(
    product_prompt_v1, 
    15, 
    "product_15_runs.txt"
)
print(f"✓ Product 15× complete. Cost: ${product_15_cost:.6f}")

extraction_15_results, extraction_15_cost = run_n_times(
    extraction_prompt_v1, 
    15, 
    "extraction_15_runs.txt"
)
print(f"✓ Extraction 15× complete. Cost: ${extraction_15_cost:.6f}")

print_cost_summary()


2G: RUNNING ALL PROMPTS 15 TIMES

▶ Starting 15-iteration tests...

▶ Running prompt 15 times: sentiment_15_runs.txt
  [1/15] ✓ Cost: $0.001710, Remaining: $0.523690
  [2/15] ✓ Cost: $0.001710, Remaining: $0.521980
